# SWI Testing

In [1]:
import brkraw
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

%matplotlib qt

class SliceViewer:
    def __init__(self, volume, dim=2):
        # volume is a 3D numpy array (x, y, z)
        self.volume = volume
        self.slices = volume.shape[2]
        self.index = self.slices // 2

        self.fig, self.ax = plt.subplots(1, 1)
        self.img = self.ax.imshow(self.volume[:, :, self.index].T, cmap="gray", origin="lower")
        self.update_title()

        self.fig.canvas.mpl_connect("scroll_event", self.onscroll)
        plt.show()

    def onscroll(self, event):
        if event.button == "up":
            if self.index < self.slices - 1:
                self.index += 1
        elif event.button == "down":
            if self.index > 0:
                self.index -= 1
        self.img.set_data(self.volume[:, :, self.index].T)
        self.update_title()
        self.fig.canvas.draw()

    def update_title(self):
        self.ax.set_title(f"Slice {self.index+1}/{self.slices}")

        # Point to your session/scan folder
scan_path = "20260210"  # adjust to the scan folder
pvdset = brkraw.load(scan_path)

pvdset.info()

visu_pars missing for scan 7 reco 3; skipping reco entry: Error mapping 'Type' with rule {'sources': [{'file': 'visu_pars', 'key': 'VisuCoreFrameType', 'reco_id': 3}]}: visu_pars


[ Study ]
   field     value
Software     Paravision v360.2.0
Operator     nmrsu
Date         2026-02-10 11:09:17.230000-05:00
ID           20260210
Number       2
InstanceUID  2.16.756.5.5.100.8323328.7777.1770739758.12951

[ Subject ]
   field     value
ID           Hans
Name         Chen
Type         Quadruped
Sex          MALE
DateOfBirth  2019-09-22 00:00:00
Weight       Unknown
Position     Head_Prone

[ Scan(s) ]
 ScanID     Method            Protocol          Dim       DimDesc        NumEchoes  TE (ms)  TR (ms)  FlipAngle (degree)
   5     Bruker:FLASH  1_Localizer_multi_slice  2    [Spatial, Spatial]  1          [4]      [15]     10                
           Shape     FOV (mm)  Num Slices  NumSlicePack         SliceOrient            ReadOrient     SliceDistance (mm)
         [192, 192]  [40, 40]  15          3             [axial, sagittal, coronal]  [L_R, H_F, H_F]  [2, 2, 2]         
         SliceGap (mm)  NumAverage  NumRepeat
         [1, 1, 1]      3           1        


## Fit Decay to CPMG

In [2]:
def fit_t2(y, et):
    # Define decay function
    def decay_fn(TE, a, b, c):
        return a * np.exp(-b * TE)

    try:
        p0 = [y.max() - y.min(), 0.01, y.min()]
        params, _ = curve_fit(decay_fn, et, y, p0=p0, maxfev=5000)
        return 1/params[1]
    except RuntimeError:
        return(np.nan)

In [3]:
from brukerapi.dataset import Dataset
dataset = Dataset('20260210/8/pdata/1/2dseq')    # create data set, works for fid, 2dseq, rawdata.x, ser
pdata_water = dataset.data/np.max(dataset.data)
dataset = Dataset('20260210/11/pdata/1/2dseq')    # create data set, works for fid, 2dseq, rawdata.x, ser
pdata_iron = dataset.data/np.max(dataset.data)

specfig = plt.figure()
te = 10.46e-3*np.arange(200)
plt.plot(te, pdata_water[np.argmax(pdata_water[:,0]),:])
plt.plot(te, pdata_iron[np.argmax(pdata_iron[:,0]),:])
plt.xlabel("Time (s)")
plt.ylabel("Normalized Amplitude")
plt.legend(["Water", "Iron"])
fit_t2(pdata_iron[np.argmax(pdata_iron[:,0]),:], te)

/tmp/ipykernel_300515/1872123030.py:8: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(decay_fn, et, y, p0=p0, maxfev=5000)


np.float64(2.2178958671255615)

In [4]:
specfig = plt.figure()
plt.plot(pdata_water[:,0])
plt.plot(pdata_iron[:,0])
plt.xlabel("Frequency Bin")
plt.ylabel("Normalized Amplitude")
plt.legend(["Water", "Iron"])
plt.show()

In [5]:
import numpy as np
raw = np.fromfile('20260210/8/rawdata.job0', dtype=np.int32)  # adjust if 16-bit
raw_complex = (raw[0::2] + 1j * raw[1::2]).reshape(200,-1)

tf = plt.figure()
plt.plot(np.abs(raw_complex[:,127]))
# plt.plot(np.abs(raw_complex[:,256]))
plt.show()

In [138]:
fit_t2(np.abs(raw_complex[:,127]), te)

/tmp/ipykernel_907724/1872123030.py:8: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(decay_fn, et, y, p0=p0, maxfev=5000)


np.float64(1.8890705583905445)

In [42]:
raw

array([ 19994,   -136,  14828, ...,   -403, -17705,   -656],
      shape=(409600,), dtype=int16)

In [115]:
tf = plt.figure()
raw_complex = (raw[0::2] + 1j * raw[1::2])
# plt.plot(np.real(raw_complex[128::512]))
# plt.plot(np.imag(raw_complex[50::512]))
plt.plot(np.real(raw_complex[0:512]))
plt.plot(np.imag(raw_complex[0:512]))
plt.show()

## Scan 6: Localizer

In [3]:
#x
scan_id = 6
niiobj1 = pvdset.convert(scan_id, reco_id=1)
viewer = SliceViewer(niiobj1[2].dataobj)

## Scan 7,10,12,13: SWI

In [2]:
#x
scan_id = 13

niiobj1 = pvdset.convert(scan_id, reco_id=1)
viewer = SliceViewer(niiobj1.dataobj)
